# MolGap K1 frozen BN mechanism on A100
Four buffer-only interventions; frozen weights; consumed internal development only.
Results saved to private Drive. Release runtime after completion.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, os, subprocess, sys, time, zipfile
DRIVE_ROOT=Path('/content/drive/MyDrive/MolGap')
RUN_ROOT=DRIVE_ROOT/'V5/runs/k1-bn-mechanism-a100-20261008/attempt-001'
RUN_ROOT.mkdir(parents=True,exist_ok=True)
print('Durable output:',RUN_ROOT,flush=True)


In [ ]:
# Immutable source/input transfer: no credentials or dataset download.
ARCHIVE=DRIVE_ROOT/'k1-bn-mechanism-a100-20261008.zip'
EXPECTED_SHA256='07488e3a05eb7046b6a0eb35a2e91f4f8135e38af3455ae9eeb2913a4accdd7f'
assert ARCHIVE.is_file(), 'Upload the prepared ZIP to MyDrive/MolGap first.'
assert hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()==EXPECTED_SHA256
PAYLOAD=Path('/content/k1-bn-mechanism-a100-20261008')
PAYLOAD.mkdir(exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as z:
    for name in z.namelist():
        target=(PAYLOAD/name).resolve()
        assert target.is_relative_to(PAYLOAD.resolve()), 'Unsafe ZIP path'
    z.extractall(PAYLOAD)
manifest=json.loads((PAYLOAD/'payload_manifest.json').read_text())
for name,digest in manifest['files'].items():
    assert hashlib.sha256((PAYLOAD/name).read_bytes()).hexdigest()==digest, name
import shutil
shutil.copy2(PAYLOAD/'payload_manifest.json',RUN_ROOT/'payload_manifest.json')
shutil.copy2(PAYLOAD/'protocol.md',RUN_ROOT/'protocol.md')
print('PAYLOAD_IDENTITY_VERIFIED',len(manifest['sample_source_idx']),flush=True)


In [ ]:
# Isolated retained Python3.11/Torch2.4.1 dependencies; notebook kernel stays intact.
setup_started=time.time()
print('Preparing isolated retained runtime',flush=True)
subprocess.run([sys.executable,'-m','pip','install','uv==0.10.9'],check=True,timeout=180)
ENV=Path('/content/k1-profile-python311')
if not (ENV/'bin/python').exists():
    subprocess.run(['uv','venv','--python','3.11',str(ENV)],check=True,timeout=180)
PYTHON=str(ENV/'bin/python')
subprocess.run(['uv','pip','install','--python',PYTHON,'torch==2.4.1',
    '--index-url','https://download.pytorch.org/whl/cu121'],check=True,timeout=480)
subprocess.run(['uv','pip','install','--python',PYTHON,'numpy==1.26.4',
    'torch-geometric==2.6.1','ogb==1.3.6'],check=True,timeout=300)
(RUN_ROOT/'setup_observation.json').write_text(json.dumps({'setup_wall_seconds':time.time()-setup_started,
    'python_environment':PYTHON,'setup_allocated_device_hours_unknown':True},sort_keys=True))
print('ISOLATED_RUNTIME_READY',flush=True)


In [ ]:
# One bounded worker. Existing result/runtime requires reconciliation, never blind rerun.
assert not (RUN_ROOT/'runtime.json').exists(), 'Reconcile existing attempt before rerun.'
env=os.environ.copy()
env['PYTHONPATH']=str(PAYLOAD/'src')
env['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
env['PYTHONHASHSEED']='42'
log=RUN_ROOT/'liveworker.log'
from threading import Timer
command=[PYTHON,'-u','-m','molgap.k1_bn_mechanism','--root',str(PAYLOAD),'--output',str(RUN_ROOT)]
with log.open('w') as handle:
    process=subprocess.Popen(command,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    watchdog=Timer(1200,process.kill)
    watchdog.start()
    worker_started=time.time()
    try:
        for line in process.stdout:
            handle.write(line); handle.flush(); print(line,end='',flush=True)
        returncode=process.wait()
    finally:
        watchdog.cancel()
        (RUN_ROOT/'worker_process_observation.json').write_text(json.dumps({
            'worker_wall_seconds_including_imports':time.time()-worker_started,
            'returncode':process.returncode,'wall_ceiling_seconds':1200},sort_keys=True))
assert returncode==0, f'Worker failed({returncode}); durable evidence: {RUN_ROOT}'
print('DURABLE_K1_BN_MECHANISM_COMPLETE',flush=True)


In [ ]:
completion=json.loads((RUN_ROOT/'completion.json').read_text())
assert completion['complete']
for name,digest in completion['files'].items():
    assert hashlib.sha256((RUN_ROOT/name).read_bytes()).hexdigest()==digest,name
result=json.loads((RUN_ROOT/'result.json').read_text())
print('Original MAE:',result['original_mae_eV'])
for case in result['cases']:
    print(case['case'],'MAE:',case['mae_eV'],'seconds:',case['case_wall_seconds'])
with zipfile.ZipFile(DRIVE_ROOT/'k1-bn-mechanism-a100-results.zip','w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in RUN_ROOT.iterdir():
        if p.is_file():z.write(p,p.name)
print('DURABLE_BN_MECHANISM_RESULTS_COMPLETE',RUN_ROOT,flush=True)
